In [31]:
import pickle
import pandas as pd
import numpy as np
import seaborn as sns

from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
%matplotlib inline

In [8]:
def train(df_X, y, learning_rate):
    features = df_X.to_dict(orient='records')
    
    dv = DictVectorizer(sparse=False)
    dv.fit(features)
    X = dv.transform(features)

    model = LogisticRegression(solver='liblinear', C=learning_rate, max_iter=1000)
    model.fit(X, y)

    return dv, model


def predict(df_X, dv, model):
    features = df_X.to_dict(orient='records')
    X = dv.transform(features)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

In [25]:
df = pd.read_csv('course_lead_scoring_2026.csv')
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('NA')
    else:
        df[col] = df[col].fillna(0)

numerical_features = []
for col in df.columns:
    if df[col].dtype != 'object' and col != 'converted':
        numerical_features.append(col)
        
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['converted']
y_val = df_val['converted']
y_text = df_test['converted']

del df_train['converted']
del df_val['converted']
del df_test['converted']

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

X_train = dv.transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
learning_rates = [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1]

best_roc = 0
best_c = None
best_model = None
best_dv = None
for c in learning_rates:
    roc_aucs = []
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_idx]
        y_train = df_train.converted.values
    
        df_val = df_full_train.iloc[val_idx]
        y_val = df_val.converted.values
    
        del df_train['converted']
        del df_val['converted']
        
        dv, model = train(df_train, y_train, c)
        y_pred = predict(df_val, dv, model)
    
        rocauc = roc_auc_score(y_val, y_pred)
        roc_aucs.append(rocauc)

    mean_rocs_kfolds = np.mean(roc_aucs)
    print(f"Model Accuracy (ROC) = {mean_rocs_kfolds:.3f} [Learning Rate={c}]")
    if mean_rocs_kfolds > best_roc:
        best_roc = mean_rocs_kfolds
        best_c = c
        best_model = model
        best_dv = dv

Model Accuracy (ROC) = 0.617 [Learning Rate=1e-06]
Model Accuracy (ROC) = 0.657 [Learning Rate=1e-05]
Model Accuracy (ROC) = 0.752 [Learning Rate=0.0001]
Model Accuracy (ROC) = 0.749 [Learning Rate=0.001]
Model Accuracy (ROC) = 0.733 [Learning Rate=0.01]
Model Accuracy (ROC) = 0.732 [Learning Rate=0.1]
Model Accuracy (ROC) = 0.732 [Learning Rate=1]


In [27]:
output_file = f"model_C={best_c}.bin"
with open(output_file, 'wb') as f_out:
    pickle.dump((best_dv, best_model), f_out)

In [28]:
with open(output_file, 'rb') as f_in:
    (best_dv, best_model) = pickle.load(f_in)

In [30]:
(best_dv, best_model)

(DictVectorizer(sparse=False),
 LogisticRegression(C=0.0001, max_iter=1000, solver='liblinear'))

In [34]:
customer = {
    'customerid': '8879-zkjof',
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'no',
    'dependents': 'no',
    'tenure': 41,
    'phoneservice': 'yes',
    'multiplelines': 'no',
    'internetservice': 'dsl',
    'onlinesecurity': 'yes',
    'onlinebackup': 'no',
    'deviceprotection': 'yes',
    'techsupport': 'yes',
    'streamingtv': 'yes',
    'streamingmovies': 'yes',
    'contract': 'one_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'bank_transfer_(automatic)',
    'monthlycharges': 79.85,
    'totalcharges': 3320.75
}
df = pd.DataFrame([customer])
y_pred = predict(df, dv, model)
y_pred[0]

0.4765427428567612

In [44]:
def predict_single(customer, dv, model):
    X = dv.transform([customer])
    y_pred = model.predict_proba(X)[0, 1]
    return y_pred

predict_single(customer, best_dv, best_model)

0.49940256951043

In [45]:
import requests
url = 'http://127.0.0.1:9000/predict'
response = requests.post(url, json=customer)
result = response.json()
result

{'Prediction': False, 'Prediction probability': 0.4994025457044954}